# Food Recognition & 3D Plate Scanning Research Pipeline
### Kaggle Cloud GPU Training & Evaluation Pipeline (NVIDIA T4 / P100)

**Source of Truth Repository:** `https://github.com/VaibhavBajpaij/food-plat-scanner-ai-model.git`  
**Primary Model:** MobileNetV3-Small (Pretrained Transfer Learning & Staged Fine-Tuning)  

This notebook provides a fully reproducible cloud training and evaluation workflow:
1. **Clones Repository**: Clones the GitHub source of truth directly into `/kaggle/working`.
2. **Hardware & Environment Audit**: Inspects CUDA accelerator, dedicated VRAM, and disk storage limits.
3. **Scientific Unit Tests**: Executes automated test suite (`python -m src.utils.run_unit_tests`).
4. **Perceptual Hashing & Deduplication**: Verifies zero duplicate images or cross-split leakage using 64-bit difference hashing (`dHash`) and SHA256 checksums.
5. **Staged Transfer Learning**: Trains custom classification head (Stage 1) followed by staged fine-tuning of upper bottleneck blocks (Stage 2) using AdamW and Cosine Annealing.
6. **Multi-Split Evaluation**: Evaluates model performance on both the untouched historical test set (25 images) and new independent test sets.
7. **Artifact Export & Verification**: Validates checkpoint reloadability and exports artifacts directly to `/kaggle/working` for instant local download.

### 1. Clone GitHub Repository (Source of Truth)

In [ ]:
import os
import sys
import subprocess

REPO_URL = "https://github.com/VaibhavBajpaij/food-plat-scanner-ai-model.git"
REPO_DIR = "/kaggle/working/food-plat-scanner-ai-model"

if os.path.exists(REPO_DIR):
    print(f"Repository already exists at {REPO_DIR}. Pulling latest commits...")
    subprocess.run(["git", "pull", "origin", "main"], cwd=REPO_DIR, check=True)
else:
    print(f"Cloning source of truth repository from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"Current Working Directory: {os.getcwd()}")

### 2. Hardware, GPU & Disk Storage Environment Audit

In [ ]:
import shutil
import torch

print("=" * 80)
print("KAGGLE RUNTIME & HARDWARE AUDIT")
print("=" * 80)
!nvidia-smi

cuda_available = torch.cuda.is_available()
print(f"\nPyTorch Version:         {torch.__version__}")
print(f"CUDA Available:          {cuda_available}")
if cuda_available:
    device_name = torch.cuda.get_device_name(0)
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Active GPU Accelerator:  {device_name}")
    print(f"Total Dedicated VRAM:    {total_mem_gb:.2f} GB")
else:
    print("WARNING: GPU is not active! Please navigate to Kaggle Settings -> Accelerator -> GPU T4 x2 or P100.")

total_disk, used_disk, free_disk = shutil.disk_usage("/kaggle/working")
print(f"\nWorking Disk Space: Total={total_disk/(1024**3):.1f} GB | Free={free_disk/(1024**3):.1f} GB")
print("=" * 80)

### 3. Run Scientific Unit & Integrity Tests

In [ ]:
!python -m src.utils.run_unit_tests

### 4. Perceptual Hashing & Split Deduplication Audit
Verifies 64-bit difference hashing (`dHash`), Hamming distance near-duplicate detection, and capture-session isolation.

In [ ]:
!python -m src.data.image_validator

### 5. Execute Kaggle GPU Staged Transfer Learning Training
Trains:
- **Stage 1**: Custom linear classification head on top of frozen MobileNetV3-Small backbone.
- **Stage 2**: Staged fine-tuning of upper bottleneck blocks 10-12 with cosine annealing learning rate scheduler.
- Saves the best validation accuracy checkpoint.

In [ ]:
!python -m src.classification.train_kaggle

### 6. Verify Checkpoint Integrity & Export for Download
Verifies that the generated checkpoint can be loaded into memory and produces expected parameters.

In [ ]:
import json
import torch

checkpoint_candidates = [
    "/kaggle/working/models/mobilenet_v3_food_kaggle_best.pth",
    "models/checkpoints/mobilenet_v3_food_kaggle_best.pth",
    "models/checkpoints/mobilenet_v3_food_finetuned_v3.pth"
]

verified_ckpt = next((p for p in checkpoint_candidates if os.path.exists(p)), None)
assert verified_ckpt is not None, "Error: No trained checkpoint found!"

size_mb = os.path.getsize(verified_ckpt) / (1024**2)
print(f"Found checkpoint: {verified_ckpt} ({size_mb:.2f} MB)")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
data = torch.load(verified_ckpt, map_location=device, weights_only=False)
assert "model_state_dict" in data, "Corrupt checkpoint!"
print(f"Checkpoint Validation: PASS (Epoch {data.get('epoch')}, Best Val Acc: {data.get('val_acc', 0.0):.1%})")

# Export to root /kaggle/working for 1-click download
if os.path.exists("/kaggle/working"):
    export_dest = "/kaggle/working/mobilenet_v3_food_kaggle_checkpoint.pth"
    shutil.copy(verified_ckpt, export_dest)
    print(f"Checkpoint successfully exported to {export_dest} for local download.")

### 7. Display Final Benchmark Metrics Report

In [ ]:
report_candidates = [
    "/kaggle/working/outputs/kaggle_training_report.json",
    "outputs/kaggle_training_report.json",
    "outputs/phase3_evaluation_report.json"
]

report_path = next((p for p in report_candidates if os.path.exists(p)), None)
if report_path:
    with open(report_path, "r", encoding="utf-8") as f:
        rep = json.load(f)

    print("=" * 95)
    print("FINAL EVALUATION METRICS REPORT")
    print("=" * 95)
    if "hardware_environment" in rep:
        print(f"Hardware Device:     {rep['hardware_environment'].get('device_name')}")
    print(f"Training Duration:   {rep.get('training_duration_seconds', 'N/A')} s")
    print(f"Checkpoint Size:     {rep.get('checkpoint_size_mb', 'N/A')} MB")

    if "evaluation_historical_test" in rep and rep["evaluation_historical_test"].get("total_samples", 0) > 0:
        h = rep["evaluation_historical_test"]
        print(f"\nHISTORICAL TEST SET (Untouched): Accuracy = {h['accuracy']:.1%}, Macro-F1 = {h['macro_f1']:.4f}")
        for cname, m in h.get("per_class", {}).items():
            print(f"  - {cname:<25}: Precision={m['precision']:.3f}, Recall={m['recall']:.3f}, F1={m['f1_score']:.3f} (n={m['support']})")

    if "evaluation_independent_test" in rep and rep["evaluation_independent_test"].get("total_samples", 0) > 0:
        ind = rep["evaluation_independent_test"]
        print(f"\nINDEPENDENT EXPANDED TEST SET: Accuracy = {ind['accuracy']:.1%}, Macro-F1 = {ind['macro_f1']:.4f}")
        for cname, m in ind.get("per_class", {}).items():
            print(f"  - {cname:<25}: Precision={m['precision']:.3f}, Recall={m['recall']:.3f}, F1={m['f1_score']:.3f} (n={m['support']})")
    print("=" * 95)